# Fingerprint biometry pipeline
Pipeline for finding fingerprint's minutiae.

In [ ]:
################### LIBRARIES ###################
import os
from pathlib import Path
import random
import re
from collections import Counter, deque
import itertools
from itertools import product, combinations
import math
from math import atan2, degrees
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import ListedColormap, BoundaryNorm

import cv2
from PIL import Image
import fingerprint_enhancer
from skimage.util import invert

def pipeline(data_path):
    
    ################### FUNCTIONS ###################
    
    # Loading ---------------------------------------------------
    def load_images(path):
        valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}
        folder = Path(path)
        if not folder.is_dir():
            raise ValueError(f"Not a directory: {path!r}")
        images = []
        labels = []
        for file in folder.iterdir():
            if file.suffix.lower() in valid_exts:
                try:
                    data = np.fromfile(str(file), dtype=np.uint8)
                    img = cv2.imdecode(data, cv2.IMREAD_GRAYSCALE)
                    if img is None:
                        print(f"Warning: failed to decode {file.name}")
                        continue
                    images.append(img)
                    labels.append(file.name)
                except Exception as e:
                    print(f"Warning: could not load {file.name} ({e})")
        return images, labels
    
    # Preprocessing ---------------------------------------------------
    
    def adaptive_background_mask(image_shape, variance_grid,
                                 variance_threshold):
        """
        Compute a mask of foreground (fingerprint) segments by:
        1. Marking segments with variance >= threshold as foreground.
        2. Flood-filling low-variance (background) segments connected to the image border.
        3. Treating remaining segments as foreground.
        """
        rows, cols = variance_grid.shape
        # initial foreground mask by threshold
        segment_foreground = variance_grid >= variance_threshold
        
        # initial background mask (segments below threshold)
        segment_background = ~segment_foreground
        
        # prepare flood-fill from border for background connectivity
        flood_mask = segment_background.copy()
        visited = np.zeros_like(flood_mask, dtype=bool)
        from collections import deque
        queue = deque()
        
        # enqueue all border background segments
        for r in range(rows):
            for c in [0, cols-1]:
                if flood_mask[r, c]:
                    queue.append((r, c))
                    visited[r, c] = True
        for c in range(cols):
            for r in [0, rows-1]:
                if flood_mask[r, c]:
                    queue.append((r, c))
                    visited[r, c] = True
        
        # 4-neighborhood
        directions = [(-1,0), (1,0), (0,-1), (0,1)]
        while queue:
            r, c = queue.popleft()
            for dr, dc in directions:
                nr, nc = r+dr, c+dc
                if 0 <= nr < rows and 0 <= nc < cols:
                    if flood_mask[nr, nc] and not visited[nr, nc]:
                        visited[nr, nc] = True
                        queue.append((nr, nc))
        
        # any background segment not visited is interior hole => treat as foreground
        segment_foreground = segment_foreground | (~visited & segment_background)
        return segment_foreground
    
    def prune_foreground_segments(
        segment_foreground,
        min_foreground_neighbors=3):
        """
        Remove isolated or sparsely connected segments
        based on neighbourhood count (up to 8).
        """
        rows, cols = segment_foreground.shape
        pruned = np.zeros_like(segment_foreground, dtype=bool)
        # 8-neighborhood offsets
        neighbors = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]
        for r in range(rows):
            for c in range(cols):
                if not segment_foreground[r, c]:
                    continue
                count = 0
                for dr, dc in neighbors:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < rows and 0 <= nc < cols and segment_foreground[nr, nc]:
                        count += 1
                if count >= min_foreground_neighbors:
                    pruned[r, c] = True
        return pruned
    
    def preprocess_images(images, segment_size,
                          variance_threshold, min_neighbors):
        
        """
        Clear background by image segmentation
        and brightness variance in segments.
        Then equalize the histogram per fingerprint segment.
        """
        preprocessed_images = []
        foreground_masks   = []
        for grayscale_image in images:
            image_height, image_width = grayscale_image.shape
            num_rows = int(np.ceil(image_height / segment_size))
            num_cols = int(np.ceil(image_width  / segment_size))
    
            # compute variance grid
            variance_values = np.zeros((num_rows, num_cols), dtype=np.float32)
            for r in range(num_rows):
                for c in range(num_cols):
                    y0, x0 = r*segment_size, c*segment_size
                    y1, x1 = min((r+1)*segment_size, image_height), min((c+1)*segment_size, image_width)
                    tile = grayscale_image[y0:y1, x0:x1]
                    variance_values[r, c] = float(np.var(tile))
    
            # threshold variance (Otsu if not provided)
            if variance_threshold is None:
                flat = cv2.normalize(variance_values, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8).flatten()
                _, otsu_thresh = cv2.threshold(flat, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
                variance_threshold = variance_values.flatten()[flat >= otsu_thresh].min()
    
            # adaptive foreground mask
            segment_fg = adaptive_background_mask(grayscale_image.shape, variance_values, variance_threshold)
    
            # prune by neighbor count
            pruned_fg = prune_foreground_segments(segment_fg, min_neighbors)
            foreground_masks.append(pruned_fg)
    
            # Stage 1: draw grid and highlight pruned foreground
            grid_image = cv2.cvtColor(grayscale_image, cv2.COLOR_GRAY2BGR)
            for r in range(num_rows):
                for c in range(num_cols):
                    y0, x0 = r*segment_size, c*segment_size
                    y1, x1 = min((r+1)*segment_size, image_height), min((c+1)*segment_size, image_width)
                    color = (0,255,0) if pruned_fg[r,c] else (0,0,255)
                    cv2.rectangle(grid_image, (x0,y0), (x1-1,y1-1), color, 1)
    
            # clean background segments
            cleaned = grayscale_image.copy()
            for r in range(num_rows):
                for c in range(num_cols):
                    if not pruned_fg[r,c]:
                        y0, x0 = r*segment_size, c*segment_size
                        y1, x1 = min((r+1)*segment_size, image_height), min((c+1)*segment_size, image_width)
                        cleaned[y0:y1, x0:x1] = 255
    
            # histogram equalization per pruned foreground segment
            equalized = cleaned.copy()
            for r in range(num_rows):
                for c in range(num_cols):
                    if pruned_fg[r,c]:
                        y0, x0 = r*segment_size, c*segment_size
                        y1, x1 = min((r+1)*segment_size, image_height), min((c+1)*segment_size, image_width)
                        seg = equalized[y0:y1, x0:x1]
                        equalized[y0:y1, x0:x1] = cv2.equalizeHist(seg)
    
            plt.show()
            preprocessed_images.append(equalized)
        return preprocessed_images, foreground_masks
    
    def enlarge_images(images, masks):
        """
        Enlarge the fingerprint region so it fills the frame as much as possible
        while preserving aspect ratio. Works even when masks are lower-res.
        """
        enlarged_imgs = []
        enlarged_msks = []
    
        for img, msk in zip(images, masks):
            # 1) Upsample mask to image size
            h_img, w_img = img.shape[:2]
            msk_up = cv2.resize(msk.astype(np.uint8), (w_img, h_img), interpolation=cv2.INTER_NEAREST)
            msk_up = (msk_up > 0).astype(np.uint8)
    
            # 2) Bounding box on upsampled mask
            ys, xs = np.nonzero(msk_up)
            if not len(xs):
                enlarged_imgs.append(img)
                enlarged_msks.append(msk_up)
                continue
            x0, x1 = xs.min(), xs.max()
            y0, y1 = ys.min(), ys.max()
    
            # 3) Crop both image and upsampled mask
            img_roi = img[y0:y1+1, x0:x1+1]
            msk_roi = msk_up[y0:y1+1, x0:x1+1]
    
            # 4) Compute scale factor
            roi_h, roi_w = img_roi.shape[:2]
            scale = min(w_img / roi_w, h_img / roi_h)
            if scale <= 1:
                enlarged_imgs.append(img)
                enlarged_msks.append(msk_up)
                continue
    
            new_w, new_h = int(roi_w * scale), int(roi_h * scale)
            img_resized = cv2.resize(img_roi, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
            msk_resized = cv2.resize(msk_roi, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
    
            # 5) Center on canvas (white background)
            pad_x = (w_img - new_w) // 2
            pad_y = (h_img - new_h) // 2
    
            # create white canvas instead of black
            canvas_img = np.full_like(img, 255)      # white background
            canvas_msk = np.zeros_like(msk_up)       # mask stays binary
    
            canvas_img[pad_y:pad_y+new_h, pad_x:pad_x+new_w] = img_resized
            canvas_msk[pad_y:pad_y+new_h, pad_x:pad_x+new_w] = msk_resized
    
            enlarged_imgs.append(canvas_img)
            enlarged_msks.append(canvas_msk)
    
        return enlarged_imgs, enlarged_msks
    
    # Binarization and cleaning ---------------------------------------------------
    
    def all_images_binary(images):
        for img in images:
            vals = np.unique(img)
            if not set(vals).issubset({0, 255}):
                return False
        return True
    
    def binarize_images(images, threshold_value=1.0):
        bin_images = []
        for image in images:
            _, binarized = cv2.threshold(image, threshold_value, 255, cv2.THRESH_BINARY)
            bin_images.append(binarized)
        return bin_images
    
    # Thinning ---------------------------------------------------
    
    def skeletonize(img, element_size=3):
        """
        Morphological skeletonization using iterative open/erode.
        img must be a binary (0/255) uint8 image.
        """
        _, img = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
        skel = np.zeros_like(img)
        element = cv2.getStructuringElement(cv2.MORPH_CROSS, (element_size, element_size))
        
        while True:
            temp = cv2.morphologyEx(img, cv2.MORPH_OPEN, element)
            temp = cv2.bitwise_not(temp)
            temp = cv2.bitwise_and(img, temp)
            skel = cv2.bitwise_or(skel, temp)
            img = cv2.erode(img, element)
            if cv2.countNonZero(img) == 0:
                break
    
        return skel
    
    A = {
        0: set([3,6,7,12,14,15,24,28,30,31,48,56,60,62,63,
                96,112,120,124,126,127,129,131,135,143,159,
                191,192,193,195,199,207,223,224,225,227,231,
                239,240,241,243,247,248,249,251,252,253,254]),
        1: set([7,14,28,56,112,131,193,224]),
        2: set([7,14,15,28,30,56,60,112,120,131,135,193,195,224,225,240]),
        3: set([7,14,15,28,30,31,56,60,62,112,120,124,131,135,143,193,195,199,224,225,227,240,241,248]),
        4: set([7,14,15,28,30,31,56,60,62,63,112,120,124,126,131,135,143,159,193,195,199,207,224,225,227,231,240,241,243,248,249,252]),
        5: set([7,14,15,28,30,31,56,60,62,63,112,120,124,126,131,135,143,159,191,193,195,199,207,224,225,227,231,239,240,241,243,248,249,251,252,254]),
    }
    n4 = [(-1,0),(1,0),(0,-1),(0,1)]
    offsets8 = [(-1,0),(-1,1),(0,1),(1,1),(1,0),(1,-1),(0,-1),(-1,-1)]
    weights8 = [1,2,4,8,16,32,64,128]
    
    def compute_weight_K3M(img, i, j):
        h, w = img.shape
        s = 0
        for (di, dj), wg in zip(offsets8, weights8):
            ni, nj = i+di, j+dj
            if 0 <= ni < h and 0 <= nj < w and img[ni, nj]==1:
                s += wg
        return s
    
    def mark_border(img):
        h, w = img.shape
        m = np.zeros_like(img, dtype=bool)
        for di, dj in n4:
            shifted = np.zeros_like(img)
            shifted[max(0,di):min(h,h+di), max(0,dj):min(w,w+dj)] = \
                img[max(0,-di):min(h,h-di), max(0,-dj):min(w,w-dj)]
            m |= (img==1) & (shifted==0)
        return m
    
    def K3M(images):
        result = []
        for bin_img in images:
            bin_img = np.where(bin_img== 0, 1, 0).astype(np.uint8)
            skel = bin_img.copy()
            h, w = skel.shape
            changed = True
    
            # 1) Iterate while changing
            while changed:
                changed = False
    
                # Phase 0: mask border pixels
                border = mark_border(skel)
    
                # Phase 1, 2, 3, 4, 5
                for phase in range(1, 6):
                    to_del = []
                    lookup = A[phase]
                    idx = np.argwhere(border & (skel==1))
                    for i, j in idx:
                        wval = compute_weight_K3M(skel, i, j)
                        if wval in lookup:
                            to_del.append((i,j))
                    if to_del:
                        changed = True
                        for i, j in to_del:
                            skel[i, j] = 0
    
            # 2) Unmark
            A1pix = A[0]
            border = mark_border(skel)
            idx = np.argwhere(border & (skel==1))
            for i, j in idx:
                if compute_weight_K3M(skel, i, j) in A1pix:
                    skel[i, j] = 0
            result.append(skel)
            print('K3M done')
    
        return result
    
    # Finding minuates ---------------------------------------------------
    
    offsets8 = [(-1,0),(-1,1),(0,1),(1,1),(1,0),(1,-1),(0,-1),(-1,-1)]
    weights8 = [1,2,4,8,16,32,64,128]
    
    # Create termination weights --------------
    termination_weights = [1,2,4,8,16,32,64,128]
    termination_orientations = []
    for w in termination_weights:
        idx = weights8.index(w)
        di, dj = offsets8[idx]
        angle = degrees(atan2(-di, dj))
        termination_orientations.append(angle)
    
    # Create bifurcation weights --------------
    def count_runs(seq):
        runs = {0: 0, 1: 0}
        current = seq[0]
        count = 1
        for i in range(1, len(seq)):
            if seq[i] != current:
                runs[current] += 1
                current = seq[i]
        runs[current] += 1
        return runs
    def is_valid_bifurcation_pattern(seq):
        if seq[0] == seq[-1]:
            runs = count_runs(seq)
            if seq[0] == 0:
                return runs[0] == 4 and runs[1] == 3
            else:
                return runs[1] == 4 and runs[0] == 3
        else:
            runs = count_runs(seq)
            return runs[0] == 3 and runs[1] == 3
    valid_weights = []
    for seq in product([0, 1], repeat=8):
        if is_valid_bifurcation_pattern(seq):
            weight = sum(w for bit, w in zip(seq, weights8) if bit)
            valid_weights.append(weight)
    bifurcation_weights = sorted(set(valid_weights))
    
    def compute_weight(img, i, j):
        h, w = img.shape
        s = 0
        for (di, dj), wg in zip(offsets8, weights8):
            ni, nj = i+di, j+dj
            if 0 <= ni < h and 0 <= nj < w and img[ni, nj]==0:
                s += wg
        return s
    
    def prune_close_points(points, min_dist):
        points = list(points)
        to_remove = set()
        n = len(points)
        for i in range(n):
            yi, xi = points[i]
            for j in range(i+1, n):
                yj, xj = points[j]
                distance = (yi-yj)**2 + (xi-xj)**2
                if distance <= min_dist**2 and distance != 0:
                    to_remove.add(i)
                    to_remove.add(j)
        return [pt for idx, pt in enumerate(points) if idx not in to_remove]
    
    
    def is_connected(skel, p1, p2, max_dist):
        """
        Check if p1->p2 are connected in the skeleton (value==0),
        allowing 8-neighbor moves, constrained to a local square of size 2*max_dist+1.
        """
        h, w = skel.shape
        (y1, x1), (y2, x2) = p1, p2
        # bounding box
        dy = max_dist; dx = max_dist
        y0 = max(0, min(y1, y2) - dy)
        y3 = min(h, max(y1, y2) + dy + 1)
        x0 = max(0, min(x1, x2) - dx)
        x3 = min(w, max(x1, x2) + dx + 1)
        sub = skel[y0:y3, x0:x3] == 0  # True where ridge
        # BFS
        start = (y1 - y0, x1 - x0)
        goal  = (y2 - y0, x2 - x0)
        visited = np.zeros_like(sub, bool)
        q = deque([start])
        visited[start] = True
        neigh8 = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
        while q:
            y, x = q.popleft()
            if (y,x) == goal:
                return True
            for dy, dx in neigh8:
                ny, nx = y+dy, x+dx
                if 0 <= ny < sub.shape[0] and 0 <= nx < sub.shape[1]:
                    if sub[ny, nx] and not visited[ny, nx]:
                        visited[ny, nx] = True
                        q.append((ny, nx))
        return False
    
    def prune_close_connected_points(points, skel, min_dist):
        """
        From points list, remove any two points closer than min_dist **and** connected by the skeleton.
        """
        points = list(points)
        to_remove = set()
        n = len(points)
        for i in range(n):
            for j in range(i+1, n):
                p1 = points[i]
                p2 = points[j]
                if (p1[0]-p2[0])**2 + (p1[1]-p2[1])**2 <= min_dist**2:
                    if is_connected(skel, p1, p2, min_dist):
                        to_remove.add(i)
                        to_remove.add(j)
        return [pt for idx, pt in enumerate(points) if idx not in to_remove]
    
    def remove_terminations_and_bifurcations_near_each_other(
        terminations, bifurcations, skel, min_dist
    ):
        """
        Remove any termination that lies within min_dist of a bifurcation AND
        is connected to it via the ridge skeleton—and also remove that bifurcation.
        Returns (new_terminations, new_bifurcations).
        """
        term_to_remove = set()
        bif_to_remove  = set()
    
        for ti, t in enumerate(terminations):
            for bi, b in enumerate(bifurcations):
                dy = t[0] - b[0]
                dx = t[1] - b[1]
                if dy*dy + dx*dx <= min_dist**2:
                    if is_connected(skel, t, b, min_dist):
                        term_to_remove.add(ti)
                        bif_to_remove.add(bi)
        new_terms = [
            t for idx, t in enumerate(terminations)
            if idx not in term_to_remove
        ]
        new_bifs  = [
            b for idx, b in enumerate(bifurcations)
            if idx not in bif_to_remove
        ]
        return new_terms, new_bifs
    
    def get_branch_orientations(skel, bif_coords, max_dist=50):
        """
        For each bifurcation at (ci,cj), follow each of its branch starters
        until an endpoint or max_dist, then compute the angle of the vector
        from (ci,cj) to that endpoint.
        Returns dict: (ci, cj) -> [angle1, angle2, angle3].
        """
        h, w = skel.shape
        orientations = {}
    
        for ci, cj in bif_coords:
            # find starters: neighbors that are skeleton pixels
            starters = [(ci+di, cj+dj) for di, dj in offsets8
                        if 0 <= ci+di < h and 0 <= cj+dj < w
                        and skel[ci+di, cj+dj] == 0]
            branch_angles = []
            for si, sj in starters:
                prev = (ci, cj)
                curr = (si, sj)
                dist = 0
                # walk along the branch
                while dist < max_dist:
                    # find skeleton neighbors of curr, excluding prev
                    neighs = []
                    i, j = curr
                    for di, dj in offsets8:
                        ni, nj = i+di, j+dj
                        if (ni, nj) != prev and 0 <= ni < h and 0 <= nj < w and skel[ni, nj] == 0:
                            neighs.append((ni, nj))
                    if len(neighs) != 1:
                        # endpoint or fork: stop here
                        break
                    # move forward
                    prev, curr = curr, neighs[0]
                    dist += 1
                end_i, end_j = curr
                # compute angle: x to right, y down; invert y for standard
                dx = end_j - cj
                dy = cj  # placeholder
                dy = end_i - ci
                angle = degrees(atan2(-dy, dx))
                branch_angles.append(angle)
            orientations[(ci, cj)] = branch_angles
        return orientations
    
    def main_bifurcation_directions(bif_dirs):
        """
        Given dict (i,j)->[angle1, angle2, angle3,...],
        returns dict (i,j)->main_angle or None.
        The main angle is the one not in the closest pair, but only if it is
        at least 45° away from both other angles; otherwise None.
        """
        def circular_diff(a, b):
            d = abs(a - b) % 360
            return min(d, 360 - d)
    
        mains = {}
        for coord, angles in bif_dirs.items():
            if len(angles) < 3:
                mains[coord] = None
                continue
            angs = angles[:3]
            # find pair with smallest angular difference
            diffs = {}
            for i in range(3):
                for j in range(i+1, 3):
                    diffs[(i, j)] = circular_diff(angs[i], angs[j])
            (i_min, j_min) = min(diffs, key=diffs.get)
            # the third index
            k = ({0, 1, 2} - {i_min, j_min}).pop()
            main_angle = angs[k]
            # check separation from the other two
            if (circular_diff(main_angle, angs[i_min]) >= 95 and
                circular_diff(main_angle, angs[j_min]) >= 95):
                mains[coord] = main_angle
            else:
                mains[coord] = None
        return mains
    
    def get_termination_orientations(skel, term_coords, max_dist=50):
        """
        For each termination at (ci,cj), follow its single branch starter
        until an endpoint or max_dist, then compute the angle of the vector
        from (ci,cj) to that endpoint.
        Returns dict: (ci, cj) -> angle (deg) or None if no clear branch.
        """
        h, w = skel.shape
        term_orients = {}
    
        for ci, cj in term_coords:
            neighbors = [(ci+di, cj+dj) for di,dj in offsets8
                         if 0 <= ci+di < h and 0 <= cj+dj < w and skel[ci+di, cj+dj]==0]
            if len(neighbors) != 1:
                term_orients[(ci, cj)] = None
                continue
    
            prev = (ci, cj)
            curr = neighbors[0]
            dist = 0
    
            # walk along the branch
            while dist < max_dist:
                i, j = curr
                next_steps = []
                for di, dj in offsets8:
                    ni, nj = i+di, j+dj
                    if (ni, nj) != prev and 0 <= ni < h and 0 <= nj < w and skel[ni, nj]==0:
                        next_steps.append((ni, nj))
                if len(next_steps) != 1:
                    break
                prev, curr = curr, next_steps[0]
                dist += 1
    
            end_i, end_j = curr
            dx = end_j - cj
            dy = end_i - ci
            
            angle = degrees(atan2(-dy, dx))
            term_orients[(ci, cj)] = angle
            
        return term_orients
    
    ################### EXECUTION ###################
    
    # Loading ---------------------------------------------------
    print('Loading images...')
    images, labels = load_images(data_path)
    # images, labels = images[:1], labels[:1]
    print(f'Loaded {len(images)} images')
    
    # Preprocessing ---------------------------------------------------
    print('Preprocessing images...')
    preprocessed_images, images_masks = preprocess_images(images,
                                            segment_size=5,
                                            variance_threshold=500,
                                            min_neighbors=6)
    print('  → preprocessing done, enlarging...')
    preprocessed_images, images_masks = enlarge_images(preprocessed_images, images_masks)
#     print('  → enlargement done, applying border cleanup and erosion...')
#     for index, mask in enumerate(images_masks):
#         mask0_u8 = (mask.astype(np.uint8)) * 255
#         mask0_u8[0, :] = False
#         mask0_u8[-1, :] = False
#         mask0_u8[:, 0] = False
#         mask0_u8[:, -1] = False
#         kernel = np.ones((15,15), dtype=np.uint8)
#         eroded_u8 = cv2.erode(mask0_u8, kernel, iterations=1)
#         eroded_mask = eroded_u8.astype(bool)
#         images_masks[index] = eroded_mask
#     print('  → mask erosion done')
    
    # Binarization and cleaning ---------------------------------------------------
    print('Binarizing and cleaning images...')
    if not all_images_binary(preprocessed_images):
        binarized_images = binarize_images(preprocessed_images, threshold_value=128)
        print('  → images binarized')
    else:
        binarized_images = preprocessed_images
        print('  → images already binary, skipping binarization')
    cleaned_images = []
    for img in binarized_images:
        inverted = cv2.bitwise_not(img)
        opened = cv2.morphologyEx(inverted, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_CROSS, (3,3)))
        result = cv2.bitwise_not(opened)
        cleaned_images.append(result)
    print(f'  → cleaning done')
    print(f'Applying Gabor filter')
    for index in range(len(cleaned_images)):
        cleaned_images[index] = np.where(fingerprint_enhancer.enhance_fingerprint(cleaned_images[index]), 0, 255).astype(np.uint8)
    for index, mask in enumerate(cleaned_images):
        mask0_u8 = (mask.astype(np.uint8)) * 255
        mask0_u8 = cv2.morphologyEx(mask0_u8, cv2.MORPH_OPEN, np.ones((45, 45), dtype=np.uint8))
        border = 15
        mask0_u8[:border, :] = 255
        mask0_u8[-border:, :] = 255
        mask0_u8[:, :border] = 255
        mask0_u8[:, -border:] = 255
        mask0_u8 = cv2.dilate(mask0_u8, np.ones((15, 15), dtype=np.uint8))
        mask0_u8 = mask0_u8.astype(bool)
        images_masks[index] = np.where(mask0_u8, False, True)
    print(f'  → gabor filter applied')
    
    # Thinning ---------------------------------------------------
    thinning_method = 'K3M'
    print(f'Applying thinning method: {thinning_method}')
    if thinning_method == 'morphological':
        skeletons = [skeletonize(cv2.bitwise_not(img)) for img in cleaned_images]
    elif thinning_method == 'K3M':
        skeletons = K3M(cleaned_images)
    else:
        raise ValueError('unknown thinning method')
    print('  → thinning done')
    for skeleton_index in range(len(skeletons)):
        skeletons[skeleton_index] = np.where(skeletons[skeleton_index] == 0, 1, 0).astype(np.uint8)
    print('  → skeleton inversion complete')

    # Finding minuates ---------------------------------------------------
    print('Detecting minutiae...')
    term_weight2orient = dict(zip(termination_weights, termination_orientations))
    images_terminations = []
    images_terminations_orientations = []
    images_bifurcations = []
    images_bifurcations_orientations = []
    for idx, skeleton in enumerate(skeletons):
        segment_mask = images_masks[idx]
        seg_h, seg_w = segment_mask.shape
        terminations = []
        bifurcations = []
        for i, j in np.argwhere(skeleton==0):
            pixel_weight = compute_weight(skeleton, i, j)
            row = i * seg_h // skeleton.shape[0]
            col = j * seg_w // skeleton.shape[1]
            if segment_mask[row, col] and pixel_weight in termination_weights:
                terminations.append((i,j))
            if pixel_weight in bifurcation_weights:
                bifurcations.append((i,j))
        images_terminations.append(terminations)
        images_bifurcations.append(bifurcations)
        print(f'  → minutiae detection: image {idx+1}/{len(skeletons)} found '
              f'{len(terminations)} terminations, {len(bifurcations)} bifurcations')
    term_term_dist = 8      # remove terminations closer than 5px to each other
    term_bif_dist  = 8     # remove terminations within 10px of any bifurcation
    bif_bif_dist  = 8       # remove bifurcations closer than 5px to each other
    print('Pruning minutiae...')
    for index, (terminations, bifurcations, skeleton) in enumerate(zip(images_terminations, images_bifurcations, skeletons)):  
        terms_A = prune_close_points(terminations, term_term_dist)
        terms_B, bifs_C = remove_terminations_and_bifurcations_near_each_other(
            terminations,
            bifurcations,
            skeleton,
            term_bif_dist)
        bifs_D = prune_close_connected_points(bifurcations, skeleton, bif_bif_dist)
        set_A = set(terms_A)
        set_B = set(terms_B)
        set_C = set(bifs_C)
        set_D = set(bifs_D)
        final_terms = list(set_A & set_B)
        final_bifs = list(set_C & set_D)
        images_terminations[index] = final_terms
        images_bifurcations[index] = final_bifs
        if index % 10 == 0 or index == len(skeletons)-1:
            print(f'  → pruning: image {index+1}/{len(skeletons)} final '
                  f'{len(final_terms)} terms, {len(final_bifs)} bifs')
    images_terminations_orientations = []
    images_bifurcations_orientations = []
    print('Computing orientations...')
    for index in range(len(skeletons)):
        term_coords = images_terminations[index]
        term_dirs = get_termination_orientations(skeletons[index], term_coords, max_dist=10)
        term_oris = [term_dirs.get(coord) for coord in images_terminations[index]]
        images_terminations_orientations.append(term_oris)
        bif_dirs = get_branch_orientations(skeletons[index], images_bifurcations[index], max_dist=10)
        main_dirs = main_bifurcation_directions(bif_dirs)
        bif_oris = [main_dirs.get(coord) for coord in images_bifurcations[index]]
        images_bifurcations_orientations.append(bif_oris)
    print('Processing complete.')

    return images_terminations, images_terminations_orientations, images_bifurcations, images_terminations_orientations, labels, skeletons, preprocessed_images

# Use

In [ ]:
data_path = 'SkanyLiniiPapilarnych_SebastianPergała'
images_terminations, images_terminations_orientations, images_bifurcations, images_bifurcations_orientations, labels, skeletons, preprocessed_images = pipeline(data_path=data_path)

# Showcase

In [ ]:
def plot_minutiae_overlays(
    images,
    images_terminations,
    images_bifurcations,
    count=15,
    images_per_row=5
):
    n = min(count, len(images))
    cols = min(images_per_row, n)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
    axes = axes.flatten()

    for idx in range(rows * cols):
        ax = axes[idx]
        if idx < n:
            img = images[idx]
            terms = images_terminations[idx]
            bifs  = images_bifurcations[idx]
            ax.imshow(img, cmap='gray', interpolation='nearest')
            if terms:
                ys, xs = zip(*terms)
                ax.scatter(xs, ys, s=50, facecolors='none', edgecolors='red')
            if bifs:
                yb, xb = zip(*bifs)
                ax.scatter(xb, yb, marker='x', c='blue', s=50)
            ax.set_title(f"{idx}")
        ax.axis('off')

    plt.tight_layout()
    plt.show()

def match_score_mixed(term1, term2, bif1, bif2, orient1, orient2, radius, orient_thresh=30):
    """
    term1, term2: termination points (x, y)
    bif1, bif2: bifurcation points (x, y)
    orient1, orient2: orientations for terminations (in degrees)
    radius: spatial matching radius
    orient_thresh: max allowed orientation difference for terminations
    """
    matched = 0
    used2_term = set()
    used2_bif = set()

    # Match terminations with orientation check
    for (x1, y1), a1 in zip(term1, orient1):
        for k, ((x2, y2), a2) in enumerate(zip(term2, orient2)):
            if k in used2_term:
                continue
            if (x1 - x2)**2 + (y1 - y2)**2 <= radius**2:
                diff = abs(a1 - a2) % 360
                diff = min(diff, 360 - diff)
                if diff <= orient_thresh:
                    matched += 1
                    used2_term.add(k)
                    break

    # Match bifurcations with position only
    for (x1, y1) in bif1:
        for k, (x2, y2) in enumerate(bif2):
            if k in used2_bif:
                continue
            if (x1 - x2)**2 + (y1 - y2)**2 <= radius**2:
                matched += 1
                used2_bif.add(k)
                break

    avg_count = (len(term1) + len(term2) + len(bif1) + len(bif2)) / 2.0
    return matched / avg_count if avg_count > 0 else 0.0

def compare_all_mixed(
    images_terms, images_term_oris,
    images_bifs,
    labels, radius, orient_thresh=30
):
    n = len(labels)
    scores = np.zeros((n, n))
    for i in range(n):
        term_i, bif_i = images_terms[i], images_bifs[i]
        ori_i = images_term_oris[i]
        for j in range(n):
            term_j, bif_j = images_terms[j], images_bifs[j]
            ori_j = images_term_oris[j]
            scores[i, j] = match_score_mixed(
                term_i, term_j, bif_i, bif_j,
                ori_i, ori_j,
                radius, orient_thresh
            )
    return pd.DataFrame(scores, index=labels, columns=labels)

def plot_match_scores(df_scores, figsize=(8, 8)):
    """
    Plot a heatmap of the matching scores DataFrame.
    """
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(df_scores.values, aspect='equal')
    # ticks
    labels = df_scores.index.tolist()
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticklabels(labels)
    
    n = len(labels)
    for k in range(0, n, 5):
        if k + 5 <= n:
            rect = patches.Rectangle((k - 0.5, k - 0.5), 5, 5,
                                     linewidth=1, edgecolor='black', facecolor='none')
            ax.add_patch(rect)
            
    # colorbar
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Match Score")
    ax.set_title("Fingerprint Minutiae Match Scores")
    ax.set_xlabel("Image")
    ax.set_ylabel("Image")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_minutiae_overlays(skeletons, images_terminations, images_bifurcations, count=50)

In [ ]:
radius = 30
orient_thresh = 15
df_scores = compare_all_mixed(
    images_terminations, images_terminations_orientations,
    images_bifurcations,
    labels,
    radius, orient_thresh
)
plot_match_scores(df_scores)